<a href="https://colab.research.google.com/github/amfei/RAG/blob/main/Sentiment%20Clasification(RAG%2BXAI).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

✅ Financial Sentiment Classification Pipeline with RAG + XAI

📥 Data Preparation
1. Loaded the Financial PhraseBank dataset using Hugging Face datasets.

2. Inspected and printed label distribution (Negative, Neutral, Positive).

3. Split the data into training (2000 samples) and test set (230 samples).

4. Balanced the training set using undersampling/oversampling to ensure equal samples per class (300 each) — to avoid model bias toward the Neutral class.

 🔍 Embedding + Retrieval Layer (RAG style)
5. Used BAAI/bge-large-en-v1.5 embedding model (chosen for its high accuracy and multilingual support).

6. Embedded training sentences with BGE and stored them in ChromaDB for semantic retrieval.

7. During inference, retrieved Top-K (5) similar sentences from ChromaDB using cosine similarity.

🧠 Sentiment Prediction
8. Used ProsusAI/finbert, a financial domain-specific BERT model, for sentiment classification.

9. Applied FinBERT on each of the retrieved sentences.

10. Aggregated predictions using confidence-weighted soft voting to determine the final label.

📊 Evaluation and Explainability
11. Calculated Accuracy, Precision, Recall, F1 Score, and printed the classification report.

12. Displayed a confusion matrix to understand class-level performance.

13. Plotted ROC AUC and Precision-Recall curves for each class (One-vs-Rest).

14. Calculated and printed optimal decision thresholds per class using F1-based selection.

⚡ Why These Choices?

* BGE-large → Chosen for retrieval quality and low latency.

* FinBERT → Tailored to financial text, performs better than general sentiment models in this domain.

* ChromaDB → Fast vector DB enabling retrieval-augmented generation (RAG) structure.

* Balanced training → Prevents class dominance and improves F1 for rare classes.

* Soft-voting + curves → Makes predictions explainable, and supports threshold optimization.



In [1]:
import torch
print("Torch CUDA Available:", torch.cuda.is_available())
print("Device:", torch.device("cuda" if torch.cuda.is_available() else "cpu"))

# Install required packages (run this once in Colab)
# !pip uninstall -y torchvision
# !pip install torchvision --no-cache-dir
# !pip install  -U chromadb  transformers datasets sentence-transformers torch

# Run this once after renaming your notebook
import sys
sys.modules.pop("torchvision", None)
sys.modules.pop("transformers", None)


In [1]:
# Fully-commented and rationale-explained version of your financial sentiment pipeline

import os
import torch
import random
import logging
import matplotlib.pyplot as plt
from collections import Counter, defaultdict
from datasets import load_dataset  # to load the financial_phrasebank dataset
from transformers import pipeline  # to use a pretrained FinBERT sentiment classifier
from sentence_transformers import SentenceTransformer  # to embed text using BGE
import chromadb  # vector store for approximate nearest neighbors
from typing import List, Tuple
import numpy as np
from sklearn.metrics import (
    classification_report, accuracy_score, f1_score, precision_score, recall_score,
    ConfusionMatrixDisplay, roc_auc_score, precision_recall_curve, auc,
    RocCurveDisplay
)

# --- Logging configuration ---
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("BalancedSentimentPipeline")

# --- Configuration ---
CONFIG = {
    # Using BAAI BGE-large model: strong multilingual, high-performing sentence embeddings
    "embedding_model": "BAAI/bge-large-en-v1.5",

    # Using FinBERT model fine-tuned on financial text for domain-specific sentiment classification
    "sentiment_model": "ProsusAI/finbert",

    # Location for persistent ChromaDB vector index
    "chroma_path": "./chroma_vectors",

    # Name of the collection within ChromaDB
    "collection_name": "financial_sentiment_balanced",

    # Automatically use GPU if available
    "device": "cuda" if torch.cuda.is_available() else "cpu",

    # Total number of examples to load from dataset
    "train_size": 2000,

    # Ensures each sentiment class contributes equally during training (prevents majority class bias)
    "samples_per_class": 300,

    # Number of samples to evaluate generalization performance
    "test_size": 230,

    # Number of nearest neighbors to retrieve per query
    "top_k": 5
}

# --- Load Embedding & Sentiment Models ---
embedding_model = SentenceTransformer(CONFIG["embedding_model"], device=CONFIG["device"])
sentiment_pipeline = pipeline("text-classification", model=CONFIG["sentiment_model"], device=0 if CONFIG["device"] == "cuda" else -1)

# --- Setup ChromaDB persistent vector store ---
chroma_client = chromadb.PersistentClient(path=CONFIG["chroma_path"])
collection = chroma_client.get_or_create_collection(name=CONFIG["collection_name"])

# --- Global variable to store evaluation dataset ---
eval_dataset = None

# --- Balancing function to ensure equal samples per class ---
def balance_training_set(dataset, samples_per_class=300):
    label_buckets = defaultdict(list)
    for i, row in enumerate(dataset):
        label_buckets[row["label"]].append(i)

    selected_indices = []
    for label, indices in label_buckets.items():
        if len(indices) >= samples_per_class:
            selected = random.sample(indices, samples_per_class)
        else:
            selected = random.choices(indices, k=samples_per_class)
        selected_indices.extend(selected)

    random.shuffle(selected_indices)
    return dataset.select(selected_indices)

# --- Index only the balanced training set into ChromaDB ---
def index_train_dataset_balanced():
    global eval_dataset

    # Load and shuffle full dataset
    full_dataset = load_dataset("financial_phrasebank", "sentences_allagree", split="train").shuffle(seed=42)

    # Display label imbalance
    print("\n📊 Full Dataset Label Distribution:")
    print(Counter(full_dataset["label"]))

    # Split into training and evaluation sets
    raw_train = full_dataset.select(range(CONFIG["train_size"]))
    eval_dataset = full_dataset.select(range(CONFIG["train_size"], CONFIG["train_size"] + CONFIG["test_size"]))

    # Balance the train set by undersampling the dominant class (Neutral)
    balanced_train = balance_training_set(raw_train, samples_per_class=CONFIG["samples_per_class"])

    print("\n✅ Balanced Train Set Distribution:")
    print(Counter(balanced_train["label"]))

    # Prepare and embed training samples
    sentences = balanced_train["sentence"]
    queries = ["Represent this sentence for retrieval: " + s for s in sentences]  # prompt format for BGE

    logger.info("Embedding balanced train set...")
    embeddings = embedding_model.encode(queries, normalize_embeddings=True, batch_size=32, show_progress_bar=True)

    # Store in vector DB
    collection.add(
        ids=[str(i) for i in range(len(sentences))],
        embeddings=embeddings,
        metadatas=[{"sentence": s} for s in sentences]
    )

    logger.info("Finished indexing balanced training data.")

# --- Retrieve K similar sentences and classify using FinBERT (vote-based) ---
def retrieve_and_predict(text: str) -> Tuple[str, List[str]]:
    query = "Represent this sentence for retrieval: " + text
    embedding = embedding_model.encode(query, normalize_embeddings=True).tolist()

    # Retrieve top-k semantically similar sentences
    results = collection.query(query_embeddings=[embedding], n_results=CONFIG["top_k"])
    retrieved = [meta["sentence"] for meta in results["metadatas"][0]]
    predictions = sentiment_pipeline(retrieved)

    label_map = {"negative": 0, "neutral": 1, "positive": 2}
    reverse_map = {0: "Negative", 1: "Neutral", 2: "Positive"}

    # Soft-voting: aggregate confidence scores from all retrieved sentences
    votes = Counter()
    for pred in predictions:
        label = label_map.get(pred["label"].lower())
        if label is not None:
            votes[label] += pred["score"]

    if not votes:
        return "Unknown", retrieved

    final_label = reverse_map[max(votes, key=votes.get)]
    return final_label, retrieved

# --- Evaluate predictions + show all curves ---
def evaluate_with_threshold_and_all_curves():
    label_map = {"negative": 0, "neutral": 1, "positive": 2}
    y_true, y_pred, y_scores = [], [], []

    for row in eval_dataset:
        sentence = row["sentence"]
        true = row["label"]

        # Embed + retrieve + predict
        query = "Represent this sentence for retrieval: " + sentence
        embedding = embedding_model.encode(query, normalize_embeddings=True).tolist()
        results = collection.query(query_embeddings=[embedding], n_results=CONFIG["top_k"])
        retrieved = [meta["sentence"] for meta in results["metadatas"][0]]
        preds = sentiment_pipeline(retrieved)

        scores = [0.0, 0.0, 0.0]
        for pred in preds:
            label = label_map.get(pred["label"].lower())
            if label is not None:
                scores[label] += pred["score"]

        # Normalize to get soft-labels
        total = sum(scores)
        probs = [s / total if total > 0 else 1/3 for s in scores]

        y_true.append(true)
        y_pred.append(np.argmax(probs))
        y_scores.append(probs)

    y_scores = np.array(y_scores)
    y_true = np.array(y_true)

    # Print metrics
    print("\n📊 Threshold-Based Evaluation Results")
    print("Accuracy:", accuracy_score(y_true, y_pred))
    print("Precision:", precision_score(y_true, y_pred, average="macro"))
    print("Recall:", recall_score(y_true, y_pred, average="macro"))
    print("F1 Score:", f1_score(y_true, y_pred, average="macro"))
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, target_names=["Negative", "Neutral", "Positive"]))

    # Show confusion matrix
    ConfusionMatrixDisplay.from_predictions(y_true, y_pred, display_labels=["Negative", "Neutral", "Positive"])
    plt.title("Confusion Matrix")
    plt.grid(False)
    plt.show()

    # Show ROC AUC curves (One-vs-Rest)
    plt.figure(figsize=(8, 6))
    for i, label in enumerate(["Negative", "Neutral", "Positive"]):
        y_true_bin = (y_true == i).astype(int)
        RocCurveDisplay.from_predictions(y_true_bin, y_scores[:, i], name=f"ROC - {label}")
    plt.title("ROC AUC Curves")
    plt.grid(True)
    plt.show()

    # Show Precision-Recall curves
    plt.figure(figsize=(8, 6))
    for i, label in enumerate(["Negative", "Neutral", "Positive"]):
        y_true_bin = (y_true == i).astype(int)
        precision, recall, _ = precision_recall_curve(y_true_bin, y_scores[:, i])
        pr_auc = auc(recall, precision)
        plt.plot(recall, precision, label=f"{label} (AUC={pr_auc:.2f})")
    plt.title("Precision-Recall Curves")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.legend()
    plt.grid(True)
    plt.show()

# --- Run full pipeline ---
index_train_dataset_balanced()

example = "The company's earnings exceeded all expectations and the stock soared."
sentiment, examples = retrieve_and_predict(example)

print(f"\n📢 Predicted Sentiment: {sentiment}")
print("🔍 Top Retrieved Sentences:")
for ex in examples:
    print(f"- {ex}")

evaluate_with_threshold_and_all_curves()


📊 Full Dataset Label Distribution:
Counter({1: 1391, 2: 570, 0: 303})

✅ Balanced Train Set Distribution:
Counter({2: 300, 1: 300, 0: 300})
Batches: 100%
 29/29 [00:07<00:00,  6.26it/s]

📢 Predicted Sentiment: Positive
🔍 Top Retrieved Sentences:
- Both the net sales and operating profit were record high in the company 's history .
- The company now estimates its net sales in 2010 to increase considerably from 2009 and its operating result to be clearly positive .
- Previously , the company anticipated its operating profit to improve over the same period .
- At the same time profit of the company increased by 10 % in H1 and reached Ls 79,000 .
- The acquisition will have an immediate positive impact on Aspocomp 's financial result .
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset

📊 Threshold-Based Evaluation Results
Accuracy: 0.7347826086956522
Precision: 0.6873773420165173
Recall: 0.8086419753086419
F1 Score: 0.705156099932414

Classification Report:
              precision    recall  f1-score   support

    Negative       0.45      0.93      0.61        27
     Neutral       0.98      0.66      0.79       145
    Positive       0.63      0.84      0.72        58

    accuracy                           0.73       230
   macro avg       0.69      0.81      0.71       230
weighted avg       0.83      0.73      0.75       230